# Learnable — ISL Model Server

Runs the pretrained **AI4Bharat OpenHands SL-GCN** model (INCLUDE dataset, 263 Indian Sign Language words) as an API with a public HTTPS URL that the Learnable web app can call.

The web app only sends body and hand landmark coordinates (75 points per frame), never camera video.

**Before running** (right-hand panel → *Session options*):
1. **Internet → On** (Kaggle needs a phone-verified account for this).
2. **Accelerator → None** is enough (a sign takes about 30 ms on CPU). A GPU also works but uses your weekly GPU quota.

Then click **Run All**, copy the `https://….trycloudflare.com` URL printed by the third cell, and paste it into **Learnable → Settings → Kaggle Model Server**.

Keep this notebook running while you use the app. The URL changes every time the notebook restarts.

In [ ]:
# Where to get the server code. Point BRANCH at your branch if the server isn't merged into main yet.
REPO_URL = "https://github.com/Vanshika-01-M/learnable.git"
BRANCH = "main"

import os, subprocess, sys

REPO_DIR = "/kaggle/working/learnable"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, REPO_DIR)

In [ ]:
!pip install -q "fastapi>=0.110" "uvicorn>=0.29"

In [ ]:
# Downloads the pretrained checkpoint (~40 MB, checksum-verified), starts the API and opens the tunnel.
from server.kaggle_runner import start

SERVER_URL = start(port=8000)

In [ ]:
# Check the public URL end to end.
import json, time, urllib.request

for attempt in range(12):
    try:
        print(json.load(urllib.request.urlopen(SERVER_URL + "/health", timeout=10)))
        break
    except Exception as err:
        print("waiting for the tunnel URL to resolve...", err)
        time.sleep(5)

In [ ]:
# Keeps the session busy and reports usage every 5 minutes.
# Interrupt this cell or stop the session to shut the server down.
from server.app import state

while True:
    print(time.strftime("%H:%M:%S"), "| signs recognised:", state["requests"], "|", SERVER_URL)
    time.sleep(300)